In [1]:
from config import PARQUET_CLEAN_DATASET

import duckdb

NUTRI_COLS = {  # noms bruts, sans guillemets SQL
    "kcal": "energy-kcal_100g",
    "lipides": "fat_100g",
    "satures": "saturated-fat_100g",
    "sucres": "sugars_100g",
    "sel": "salt_100g",
    "fibres": "fiber_100g",
    "proteines": "proteins_100g",
}

def _build_filter(nom: str | None, code: str | None) -> str:
    if code:
        return f"code = {code}"
    return f"strip_accents(product_name) ILIKE '%' || strip_accents('{nom}') || '%'"

def find_product(name : str | None = None, code: str | None = None, limit : int = 3) -> dict:
    """
    Recherche un produit dans le fichier parquet propre
    :param name: nom du produit
    :param code: code du produit
    :param limit: nombre de produits maximum, entre 1 et 5
    :return:
    """
    res = {
        "status" : "introuvable",
        "total" : 0,
        "products" : []
    }
    if name is None and code is None and 0 < limit < 6:
        return res
    where_statement = _build_filter(name, code)

    struct_fields = ", ".join(
        f'{k} := CASE WHEN "{col}_empty" THEN NULL ELSE "{col}" END'
        for k, col in NUTRI_COLS.items()
    )
    any_missing = " OR ".join(f'"{col}_empty"' for col in NUTRI_COLS.values())

    con = duckdb.connect()
    response = con.execute(
        f"""
        SELECT code::VARCHAR as code,
        product_name as name,
        CASE WHEN
            lower(nutriscore_grade) IN ('a','b','c','d','e')
            THEN upper(nutriscore_grade) END AS nutriscore ,
            struct_pack({struct_fields}) AS nutrients,
            (nutriscore IS NULL OR {any_missing}) AS incomplete_data
        FROM read_parquet('{PARQUET_CLEAN_DATASET}')
        where {where_statement}
        limit {limit}
        """
    ).df()
    products = []
    total = len(response)
    res["total"] = total
    if total == 0:
        return res
    elif total == 1:
        res["status"] = "ok"
    else:
        res["status"] = "ambiguous"
    for row in response.values:
        products.append({
            "code" : row[0],
            "name" : row[1],
            "nutriscore" : row[2],
            "nutrients" : row[3],
            "incomplete_data" : row[4]
        })
    res["products"] = products

    return res

In [2]:
find_product(name = "Eclairs") # 0633148599305

{'status': 'ambiguous',
 'total': 3,
 'products': [{'code': '0000241013128',
   'name': 'Eclairs',
   'nutriscore': 'D',
   'nutrients': {'kcal': 237.0,
    'lipides': 9.5,
    'satures': 4.199999809265137,
    'sucres': 21.700000762939453,
    'sel': 0.36000001430511475,
    'fibres': None,
    'proteines': 4.599999904632568},
   'incomplete_data': True},
  {'code': '00395380',
   'name': '2 éclairs au chocolat',
   'nutriscore': 'E',
   'nutrients': {'kcal': 390.6666564941406,
    'lipides': 28.0,
    'satures': 14.666666984558105,
    'sucres': 21.33333396911621,
    'sel': 0.2666666805744171,
    'fibres': 1.7333333492279053,
    'proteines': 5.066666603088379},
   'incomplete_data': False},
  {'code': '0200219013055',
   'name': 'Éclairs chocolat',
   'nutriscore': 'D',
   'nutrients': {'kcal': 310.0,
    'lipides': 18.0,
    'satures': 5.900000095367432,
    'sucres': 20.0,
    'sel': 0.47999998927116394,
    'fibres': None,
    'proteines': 4.0},
   'incomplete_data': True}]}

In [3]:
from openai import OpenAI
import requests
import json

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

modeles_installes = [m["name"] for m in requests.get("http://localhost:11434/api/tags").json()["models"]]
print(modeles_installes)

['qwen3:4b', 'qwen3:8b']


In [4]:
#Variables globales du modèle
MAX_TOKENS = 2000
URL = "http://localhost:11434/api/chat"
TEMPERATURE = 0.5
functions = {"find_product": find_product}


In [5]:
def call_llm(client : OpenAI, context: list[dict[str,str]], tools = None, max_iter = 3) -> str :
    for _ in range(max_iter):
        client_response = client.chat.completions.create(model=MODELE, messages=context, tools=tools, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, reasoning_effort="none")
        msg = client_response.choices[0].message
        if not msg.tool_calls :
            res = msg.content
            return res if res is not None else ""
        context.append(msg.to_dict())
        for call in msg.tool_calls :
            args = json.loads(call.function.arguments)
            function_name = call.function.name
            res = functions[function_name](**args)
            context.append(message_outil(function_name, call.id, res))
    return "Je n'ai pas trouvé la réponse à votre demande, veuillez reformuler"

In [6]:
from pathlib import Path
from llm.config_assistant import PROMPT_SYSTEME, MODELE, message_outil

messages = [{"role": "system", "content": PROMPT_SYSTEME}]

def talk_to_llm(tools) :
    cpt = 0
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    while cpt < 5 :
        user = {"role": "user", "content": input(f"\n Vous > ").strip()}
        print(f"\n Vous > ", user["content"])
        messages.append(user)
        llm_response = call_llm(client=client, context=messages, tools=tools)
        print(f"\n {MODELE} >",llm_response, end=" ", )
        assitant_resp = {"role": "assistant", "content": llm_response}
        messages.append(assitant_resp)
        cpt += 1

In [7]:
from llm.config_assistant import QUESTIONS_TEST, OUTILS


def test_llm_resistance() :
    for prompt, attendu in QUESTIONS_TEST :
        messages = [{"role": "system", "content": PROMPT_SYSTEME}, {"role": "user", "content": prompt}]
        print("Question : ", prompt)
        print("Réponse du LLM : ",call_llm(client=client, context=messages, tools= OUTILS))
        print("Attendu : ", attendu)

test_llm_resistance()

Question :  Quel est le Nutri-Score des Eclairs ?
Réponse du LLM :  Il existe plusieurs produits nommés "Eclairs" dans le catalogue, avec les Nutri-Scores suivants :

1. **Eclairs** : Nutri-Score **D**
   - Énergie : 237 kcal
   - Lipides : 9.5 g
   - Acides gras saturés : 4.2 g
   - Sucres : 21.7 g
   - Sel : 0.36 g
   - Fibres : non renseigné
   - Protéines : 4.6 g

2. **2 éclairs au chocolat** : Nutri-Score **E**
   - Énergie : 390.7 kcal
   - Lipides : 28 g
   - Acides gras saturés : 14.7 g
   - Sucres : 21.3 g
   - Sel : 0.27 g
   - Fibres : 1.7 g
   - Protéines : 5.1 g

3. **Éclairs chocolat** : Nutri-Score **D**
   - Énergie : 310 kcal
   - Lipides : 18 g
   - Acides gras saturés : 5.9 g
   - Sucres : 20 g
   - Sel : 0.48 g
   - Fibres : non renseigné
   - Protéines : 4 g

Si vous souhaitez comparer ou obtenir des alternatives, précisez le produit que vous avez en tête.
Attendu :  1 appel {'nom': ...} ; note et valeurs recopiées de la fiche
Question :  Compare les Eclairs et les